## The purpose of this file is to find a better model for query generation part.

In [2]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
# to reduce long log output
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
import numpy as np
from langchain_ollama import ChatOllama
from rag_flow import RAGFlow
import pandas as pd
from time import perf_counter
import requests
from sentence_transformers import SentenceTransformer


In [2]:
response = requests.get("http://localhost:11434/api/tags")
models = response.json()["models"]

for model in models:
    print(f"{model['name']}: {model['size'] / 1024**3:.2f} GB")

qwen3:8b: 4.87 GB
llama3.2:3b: 1.88 GB
nomic-embed-text:latest: 0.26 GB
gpt-oss:20b: 12.85 GB


In [6]:
evaluation_df = pd.DataFrame({
    "qwen": {
            "mean_query_relevancy": None,
            "retrieval_success_rate": None,
            "mean_latency": None
    },
    "gpt": {
        "mean_query_relevancy": None,
        "retrieval_success_rate": None,
        "mean_latency": None
    },
    "llama": {
        "mean_query_relevancy": None,
        "retrieval_success_rate": None,
        "mean_latency": None
    }
})

In [7]:
df = pd.read_csv("test_data/arxiv_classifier_300_questions.csv")
questions = df[df["label"] == "arxiv"]["question"]
questions.head(3)


0    What are the main limitations of economic grow...
2    What are the main limitations of operating sys...
3    How can researchers evaluate optical interfere...
Name: question, dtype: str

In [8]:
qwen = ChatOllama(
    model="qwen3:8b",
    base_url="http://localhost:11434"
)

gpt = ChatOllama(
    model="gpt-oss:20b",
    base_url="http://localhost:11434"
)
llama = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
)
classifier = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=15,
    reasoning=False,
    num_ctx=3000,
)


In [3]:
models = {
    "qwen": qwen,
    "gpt": gpt,
    "llama": llama,}
eval_model = SentenceTransformer("all-MiniLM-L6-v2")
latencies = {model: [] for model in models}
relevance_scores = {model: [] for model in models}
# 1 success 0 not found (timeout)
# Please note, 0 might also indicate that the system was not able to connect to the arxiv
# web page, which can happen for multiple reasons, so 0 is not an exclusive indicator of a query that
# was not able to find a title on the arxive search page
retrieval_statuses = {model: [] for model in models}

for model_name, model in models.items():
    params = {
        "query_model": model, # we don't care about these params, This is just a placeholder.
        "answer_model": llama, # we don't care about these params, This is just a placeholder
        "titles_limit": 5,
        "top_k_titles": 5,
        "chunk_size": 200,
        "chunk_overlap_fraction": 0.2,
        "top_k_chunks": 5,
        "candidate_k": 20,
        "classifier": classifier,
    }
    rag = RAGFlow(params)
    for question in questions:
        start = perf_counter()
        query = rag._generate_arxiv_query(question)
        latency = perf_counter() - start
        latencies[model_name].append(latency)
        embeddings = eval_model.encode(
            [question, query],
            normalize_embeddings=True
        )
        similarity = np.dot(
            embeddings[0],
            embeddings[1]
        )
        relevance_scores[model_name].append(float(similarity))
        top_titles = rag._retrieve_top_k_titles(query, question)
        if len(top_titles) > 0:
            retrieval_statuses[model_name].append(1)
        else:
            retrieval_statuses[model_name].append(0)

In [31]:
evaluation_df = pd.DataFrame({
    model: {
        "query_relevance": np.mean(relevance_scores[model]),
        "query_latency": np.mean(latencies[model]),
        "retrieval_success_rate": np.mean(retrieval_statuses[model]),
    }
    for model in models
})

evaluation_df

,qwen,gpt,llama
query_relevance,0.790005,0.767474,0.602592
query_latency,47.248450,4.340309,0.239298
retrieval_success_rate,0.993333,0.980000,0.831461


```
	qwen	gpt	llama
query_relevance	0.790005	0.767474	0.602592
query_latency	47.248450	4.340309	0.239298
retrieval_success_rate	0.993333	0.980000	0.831461
```
The results above show that Llama is the quickest model, however retrieval success rate is the least, which means only aprox 80% of the queries that were generated resulted in found titles. GPT  model's retrieval success rate is 98% whihc is slightly worse than Qwen's. In terms of Query relevance (cosine similarity), Llama showed the worst result - 60%. Since quality query generation is an important step but the transient one, I would prefer a quicker model for it. I will opt out of using Qwen for this purpose and try to twick GPT and Llama and compare results.

In [34]:
evaluation_df["adj_llama"] = np.nan
evaluation_df["adj_gpt"] = np.nan


In [4]:
adj_gpt = ChatOllama(
    model="gpt-oss:20b",
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=2000,
    reasoning=False
)
adj_llama = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=2000,
    reasoning=False
)
models = {
    "gpt": adj_gpt,
    "llama": adj_llama, }
eval_model = SentenceTransformer("all-MiniLM-L6-v2")
latencies = {model: [] for model in models}
relevance_scores = {model: [] for model in models}
# 1 success 0 not found (timeout)
# Please note, 0 might also indicate that the system was not able to connect to the arxiv
# web page, which can happen for multiple reasons, so 0 is not an exclusive indicator of a query that
# was not able to find a title on the arxive search page
retrieval_statuses = {model: [] for model in models}

for model_name, model in models.items():
    params = {
        "query_model": model,  # we don't care about these params, This is just a placeholder.
        "answer_model": llama,  # we don't care about these params, This is just a placeholder
        "titles_limit": 5,
        "top_k_titles": 5,
        "chunk_size": 200,
        "chunk_overlap_fraction": 0.2,
        "top_k_chunks": 5,
        "candidate_k": 20,
        "classifier": classifier,
    }
    rag = RAGFlow(params)
    for question in questions:
        start = perf_counter()
        query = rag._generate_arxiv_query(question)
        latency = perf_counter() - start
        latencies[model_name].append(latency)
        embeddings = eval_model.encode(
            [question, query],
            normalize_embeddings=True
        )
        similarity = np.dot(
            embeddings[0],
            embeddings[1]
        )
        relevance_scores[model_name].append(float(similarity))
        top_titles = rag._retrieve_top_k_titles(query, question)
        if len(top_titles) > 0:
            retrieval_statuses[model_name].append(1)
        else:
            retrieval_statuses[model_name].append(0)

In [38]:
for model_name in models:
    col = f"adj_{model_name}"

    evaluation_df.loc["query_relevance", col] = np.mean(relevance_scores[model_name])
    evaluation_df.loc["query_latency", col] = np.mean(latencies[model_name])
    evaluation_df.loc["retrieval_success_rate", col] = np.mean(retrieval_statuses[model_name])

display(evaluation_df)

,qwen,gpt,llama,adj_llama,adj_gpt
query_relevance,0.790005,0.767474,0.602592,0.523844,0.780268
query_latency,47.248450,4.340309,0.239298,0.224170,5.357353
retrieval_success_rate,0.993333,0.980000,0.831461,0.640000,0.973333


Specifying parameters has not changed the results much. I do not think it is worth searchinf for the best context window size and other numeric params. I think I will make a choice in favour of a quicker model - Llama with default params